# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import math
import random
import re
import sys
import unicodedata
import numpy as np
import pandas as pd

# Preprocesamiento
TOKEN_RE = re.compile(r"[a-z]+(?:['\-][a-z]+)*")

def clean_special(text: str) -> str:
    # Normaliza comillas, guiones largos y quita acentos
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = re.sub(r"[—–_]+", " ", text)
    text = unicodedata.normalize("NFD", text)
    return "".join(c for c in text if unicodedata.category(c) != "Mn")

def preprocess(text: str) -> list[str]:
    # Minúsculas, limpieza y tokenización
    text = text.lower()
    text = clean_special(text)
    return TOKEN_RE.findall(text)

In [2]:
CORPUS_DIR = Path("txt-files") / "epub"
MAX_DOCS = 1000
SEMILLA = 42
MIN_TOKENS = 100

# Encontrar encabezado y pie de licencia
INICIO_RE = re.compile(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*\n", re.IGNORECASE)
FIN_RE = re.compile(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK", re.IGNORECASE)

def leer_texto(ruta: Path) -> str:
    datos = ruta.read_bytes()
    cod = "latin-1" if ruta.name.lower().endswith("-8.txt") else "utf-8"
    return datos.decode(cod, errors="replace")

def extraer_titulo(texto: str, libro_id: int) -> str:
    m = re.search(r"^\s*Title:\s*(.+)", texto[:5000], re.MULTILINE)
    return m.group(1).strip() if m else f"Documento {libro_id}"

def quitar_encabezado_pie(texto: str) -> str:
    inicio, fin = INICIO_RE.search(texto), FIN_RE.search(texto)
    if not inicio or not fin or fin.start() <= inicio.end():
        return texto
    return texto[inicio.end():fin.start()]

#  Tabla de los documentos
def cargar_tabla(documentos_validos, max_docs=MAX_DOCS, semilla=SEMILLA, min_tokens=MIN_TOKENS):
    ids = sorted(documentos_validos)
    random.Random(semilla).shuffle(ids)
    ids = sorted(ids[:max_docs])

    filas, excluidos = [], []
    for libro_id in ids:
        completo = leer_texto(documentos_validos[libro_id])
        cuerpo = quitar_encabezado_pie(completo)
        tokens = [sys.intern(t) for t in preprocess(cuerpo)]
        if len(tokens) < min_tokens:
            excluidos.append(libro_id)
            continue
        filas.append((str(libro_id), extraer_titulo(completo, libro_id), cuerpo, tokens))
    tabla = pd.DataFrame(filas, columns=["doc_id", "titulo", "texto", "tokens"])
    return tabla, excluidos

with open("documentos_validos.json", encoding="utf-8") as f:
    documentos_validos = {int(k): CORPUS_DIR / v for k, v in json.load(f).items()}

df_corpus, excluidos = cargar_tabla(documentos_validos)
print(f"{len(df_corpus)} documentos | excluidos (menos de {MIN_TOKENS} tokens): {len(excluidos)} {excluidos}")

display(df_corpus.assign(texto=df_corpus["texto"].str[:80] + "...",
                         tokens=df_corpus["tokens"].map(len)).head())

989 documentos | excluidos (menos de 100 tokens): 11 [4577, 7337, 23816, 23878, 23950, 23951, 24225, 25229, 26904, 31978, 54784]


,doc_id,titulo,texto,tokens
0,10,The King James Version of the Bible,The Old Testament of the King James Version of...,790393
1,329,Island Nights' Entertainments,\r\n[Illustration]\r\n\r\n\r\n\r\n\r\nIsland N...,49166
2,369,The outlaw of Torn,"\r\n\r\n\r\n\r\nProduced by Judith Boss, and D...",65508
3,391,The Song of Roland,\r\n\r\n\r\n\r\nProduced by Douglas B. Killing...,32383
4,434,The Circular Staircase,\r\n[Illustration]\r\n\r\n\r\n\r\n\r\nThe Circ...,70416


In [3]:
N = len(df_corpus)

# Calculo DF
def calcular_df(lista_tokens):
    df = Counter()
    for tokens in lista_tokens:
        df.update(set(tokens))
    return df

# Calculo  IDF
def calcular_idf_tfidf(df, N):
    return {t: math.log(N / d) + 1 for t, d in df.items()}

# Calculo  IDF de bm25
def calcular_idf_bm25(df, N):
    return {t: math.log(1 + (N - d + 0.5) / (d + 0.5)) for t, d in df.items()}

df = calcular_df(df_corpus["tokens"])
idf_tfidf = calcular_idf_tfidf(df, N)
idf_bm25 = calcular_idf_bm25(df, N)

print(f"N = {N} documentos | vocabulario = {len(df):,} términos")
for t in ["the", "ship", "evolution", "captain"]:
    print(f"{t:<10} df={df[t]:>4}  idf_tfidf={idf_tfidf[t]:.4f}  idf_bm25={idf_bm25[t]:.4f}")

N = 989 documentos | vocabulario = 1,171,922 términos
the        df= 878  idf_tfidf=1.1190  idf_bm25=0.1195
ship       df= 457  idf_tfidf=1.7720  idf_bm25=0.7719
evolution  df= 156  idf_tfidf=2.8468  idf_bm25=1.8446
captain    df= 433  idf_tfidf=1.8260  idf_bm25=0.8258


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [4]:
from scipy.sparse import csr_matrix, diags
from sklearn.feature_extraction.text import CountVectorizer

# Matriz TF dispersa
contador = CountVectorizer(lowercase=False, token_pattern=r"\S+")
docs_texto = [" ".join(t) for t in df_corpus["tokens"]]
TF = contador.fit_transform(docs_texto).tocsr()
del docs_texto

vocab = contador.get_feature_names_out()
indice_vocab = contador.vocabulary_
doc_ids = df_corpus["doc_id"].tolist()
assert len(vocab) == len(df), "El vocabulario no coincide con el DF del Paso 1"

longitudes = np.asarray(TF.sum(axis=1)).ravel()   # |d| en tokens
avgdl = longitudes.mean()                         # longitud media

# Vectores de IDF alineados con las columnas
idf_tfidf_vec = np.array([idf_tfidf[t] for t in vocab])
idf_bm25_vec = np.array([idf_bm25[t] for t in vocab])

# TF-IDF disperso
X_tfidf = (TF @ diags(idf_tfidf_vec)).tocsr()
normas_docs = np.sqrt(np.asarray(X_tfidf.multiply(X_tfidf).sum(axis=1)).ravel())

TF_csc = TF.tocsc()
print(f"TF: {TF.shape} | elementos no nulos: {TF.nnz:,} | avgdl = {avgdl:,.1f}")

TF: (989, 1171922) | elementos no nulos: 6,735,655 | avgdl = 63,308.1


In [ ]:
consultas = {
    "Q1": ("love",
           "Textos sobre el amor: consulta de un solo término"),
    "Q2": ("the election of the new president",
           "Textos sobre elecciones presidenciales."),
    "Q3": ("ship captain storm sea",
           "Relatos de viajes y aventuras en el mar."),
    "Q4": ("evolution natural selection",
           "Textos científicos sobre la evolución de las especies."),
    "Q5": ("alchemy philosopher stone",
           "Libros sobre alquimia y la piedra filosofal."),
    "Q6": ("murder detective",
           "Novelas de misterio policial."),
    "Q7": ("the history of the roman empire and its emperors in ancient times",
           "Historia de Roma; consulta larga con muchas palabras de relleno."),
    "Q8": ("whale harpoon",
           "Textos de caza de ballenas"),
}

def clasificar_df(d, N):
    # Etiqueta según la proporción de documentos que contienen el término
    p = d / N
    if p >= 0.5:
        return "frecuente"
    if p <= 0.05:
        return "raro"
    return "medio"

filas = []
for id_q, (texto, necesidad) in consultas.items():
    tokens = preprocess(texto)
    detalle = ", ".join(f"{t} (df={df.get(t, 0)}, {clasificar_df(df.get(t, 0), N)})"
                        for t in dict.fromkeys(tokens))   # términos únicos, en orden
    filas.append((id_q, texto, len(tokens), necesidad, detalle))

tabla_consultas = pd.DataFrame(filas, columns=["Consulta", "Texto", "Nº tokens", "Qué busca", "Términos (df)"])
pd.set_option("display.max_colwidth", None)
display(tabla_consultas)

,Consulta,Texto,Nº tokens,Qué busca,Términos (df)
0,Q1,love,1,Textos sobre el amor: consulta de un solo término,"love (df=639, frecuente)"
1,Q2,the election of the new president,6,Textos sobre elecciones presidenciales.,"the (df=878, frecuente), election (df=216, medio), of (df=861, frecuente), new (df=769, frecuente), president (df=344, medio)"
2,Q3,ship captain storm sea,4,Relatos de viajes y aventuras en el mar.,"ship (df=457, medio), captain (df=433, medio), storm (df=479, medio), sea (df=626, frecuente)"
3,Q4,evolution natural selection,3,Textos científicos sobre la evolución de las especies.,"evolution (df=156, medio), natural (df=669, frecuente), selection (df=258, medio)"
4,Q5,alchemy philosopher stone,3,Libros sobre alquimia y la piedra filosofal.,"alchemy (df=36, raro), philosopher (df=228, medio), stone (df=582, frecuente)"
5,Q6,murder detective,2,Novelas de misterio policial.,"murder (df=381, medio), detective (df=85, medio)"
6,Q7,the history of the roman empire and its emperors in ancient times,12,Historia de Roma; consulta larga con muchas palabras de relleno.,"the (df=878, frecuente), history (df=616, frecuente), of (df=861, frecuente), roman (df=415, medio), empire (df=338, medio), and (df=848, frecuente), its (df=764, frecuente), emperors (df=88, medio), in (df=914, frecuente), ancient (df=549, frecuente), times (df=731, frecuente)"
7,Q8,whale harpoon,2,Caza de ballenas; términos muy raros.,"whale (df=81, medio), harpoon (df=14, raro)"


In [6]:
K1, B = 1.2, 0.75

    # Ignora terminos fuera del vocabulario
def terminos_en_vocabulario(tokens):
    return [t for t in tokens if t in indice_vocab]

    # Orden del top considerando solo score > 0; orden descendente y, en empate, doc_id ascendente
def ordenar_top(scores, k=10):
    candidatos = np.flatnonzero(scores > 0)
    ordenados = sorted(candidatos, key=lambda i: (-scores[i], doc_ids[i]))
    return [(doc_ids[i], float(scores[i])) for i in ordenados[:k]]

    # Coseno entre el vector TF-IDF de la consulta y cada documento.
def ranking_tfidf(tokens, k=10):
    conteos = Counter(terminos_en_vocabulario(tokens))
    if not conteos:
        # caso sin términos
        return []                                       
    
    cols = [indice_vocab[t] for t in conteos]
    vals = [f * idf_tfidf_vec[indice_vocab[t]] for t, f in conteos.items()]
    q = csr_matrix((vals, ([0] * len(cols), cols)), shape=(1, len(vocab)))
    productos = (X_tfidf @ q.T).toarray().ravel()
    norma_q = np.sqrt(sum(v * v for v in vals))
    return ordenar_top(productos / (normas_docs * norma_q), k)

def aporte_bm25(j, k1=K1, b=B):
    # idf * f*(k1+1) / (f + k1*(1 - b + b*|d|/avgdl))
    col = TF_csc[:, j]
    docs, f = col.indices, col.data
    denominador = f + k1 * (1 - b + b * longitudes[docs] / avgdl)
    return docs, idf_bm25_vec[j] * f * (k1 + 1) / denominador

def scores_bm25(tokens, k1=K1, b=B):
    # suma de los aportes de cada término distinto de la consulta
    scores = np.zeros(N)
    for t in sorted(set(terminos_en_vocabulario(tokens))):
        docs, aporte = aporte_bm25(indice_vocab[t], k1, b)
        scores[docs] += aporte
    return scores

def ranking_bm25(tokens, k=10, k1=K1, b=B):
    return ordenar_top(scores_bm25(tokens, k1, b), k)

In [7]:
titulo_de = dict(zip(df_corpus["doc_id"], df_corpus["titulo"]))

# rankings[id_q] = {"tfidf": [(doc_id, score)...], "bm25": [...]}
rankings = {}   
filas = []
for id_q, (texto, _) in consultas.items():
    tokens = preprocess(texto)
    rankings[id_q] = {"tfidf": ranking_tfidf(tokens), "bm25": ranking_bm25(tokens)}
    t, m = rankings[id_q]["tfidf"], rankings[id_q]["bm25"]
    filas.append((id_q, len(t), len(m),
                  f"{t[0][0]} {titulo_de[t[0][0]][:30]}" if t else "-",
                  f"{m[0][0]} {titulo_de[m[0][0]][:30]}" if m else "-"))

display(pd.DataFrame(filas, columns=["Consulta", "Tamaño TF-IDF", "Tamaño BM25",
                                      "Primero TF-IDF", "Primero BM25"]))

def mostrar_lista(id_q, metodo):
    # Imprime el top-10 de un método: posición, ID, score y título
    print(f"{id_q} ({metodo}): {consultas[id_q][0]}")
    for pos, (doc_id, score) in enumerate(rankings[id_q][metodo], start=1):
        print(f"  {pos:>2}. {doc_id:<7}{score:>10.4f}  {titulo_de[doc_id]}")

mostrar_lista("Q3", "tfidf")
mostrar_lista("Q3", "bm25")

,Consulta,Tamaño TF-IDF,Tamaño BM25,Primero TF-IDF,Primero BM25
0,Q1,10,10,1041 Shakespeare's Sonnets,1041 Shakespeare's Sonnets
1,Q2,10,10,65442 Disunion and Restoration in Te,27348 The 2007 CIA World Factbook
2,Q3,10,10,70148 Time for survival,21749 Man on the Ocean: A Book about
3,Q4,10,10,53639 Last Words on Evolution: A Pop,59171 Degeneration: A Chapter in Dar
4,Q5,10,10,65794 Primer of Ohio Archaeology: Th,"23100 A Budget of Paradoxes, Volume"
5,Q6,10,10,59428 The Florentine Dagger: A Novel,59428 The Florentine Dagger: A Novel
6,Q7,10,10,35461 A Short History of the World,6839 The Old Roman World : the Gran
7,Q8,10,10,75370 David goes to Greenland,75370 David goes to Greenland


Q3 (tfidf): ship captain storm sea
   1. 70148      0.1135  Time for survival
   2. 72974      0.0767  Recollections of James Anthony Gardner, commander R.N. (1775–1814)
   3. 40580      0.0641  A general history of the pyrates
   4. 50908      0.0467  A Relic of the Revolution
   5. 4994       0.0449  Five Thousand Miles Underground; Or, the Mystery of the Centre of the Earth
   6. 28437      0.0449  It's a Small Solar System
   7. 59309      0.0422  Forced Move
   8. 35427      0.0402  The Raid of John Brown at Harper's Ferry as I Saw It
   9. 7509       0.0389  The Logbooks of the Lady Nelson
  10. 21749      0.0388  Man on the Ocean: A Book about Boats and Ships
Q3 (bm25): ship captain storm sea
   1. 21749      5.9910  Man on the Ocean: A Book about Boats and Ships
   2. 14172      5.9679  Willis the Pilot : A Sequel to the Swiss Family Robinson
   3. 14149      5.9532  The Pilots of Pomona: A Story of the Orkney Islands
   4. 25993      5.9336  With Cochrane the Dauntless
   5. 5

In [8]:
# 1) Consulta sin términos del vocabulario produce dos listas vacías
tokens_vacia = preprocess("qwzxv plgkt")
lista_tfidf = ranking_tfidf(tokens_vacia)
lista_bm25 = ranking_bm25(tokens_vacia)

if lista_tfidf == [] and lista_bm25 == []:
    print("Consulta fuera del vocabulario: dos listas vacías")
else:
    print("✗ Error: TF-IDF =", lista_tfidf, "| BM25 =", lista_bm25)

# 2) BM25 calculado a mano para un documento coincide con la función
tokens_q = preprocess("ship captain storm sea")
scores = scores_bm25(tokens_q)
i = int(np.argmax(scores))
tokens_doc = df_corpus["tokens"].iloc[i]
conteos_doc = Counter(tokens_doc)
manual = sum(idf_bm25[t] * conteos_doc[t] * (K1 + 1)
             / (conteos_doc[t] + K1 * (1 - B + B * len(tokens_doc) / avgdl))
             for t in set(tokens_q) if conteos_doc[t] > 0)
print(f"BM25 manual = {manual:.6f} | función = {scores[i]:.6f}")
assert abs(manual - scores[i]) < 1e-9

Consulta fuera del vocabulario: dos listas vacías
BM25 manual = 5.991042 | función = 5.991042


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [9]:
# Tabla de comparacion con Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25
def tabla_comparacion(id_q):
    t, m = rankings[id_q]["tfidf"], rankings[id_q]["bm25"]
    filas = []
    for pos in range(max(len(t), len(m))):
        izq = t[pos] if pos < len(t) else None
        der = m[pos] if pos < len(m) else None
        filas.append((pos + 1,
                      f"{izq[0]} - {titulo_de[izq[0]][:35]}" if izq else "-",
                      round(izq[1], 4) if izq else None,
                      f"{der[0]} - {titulo_de[der[0]][:35]}" if der else "-",
                      round(der[1], 4) if der else None))
    return pd.DataFrame(filas, columns=["Posición", "ID y título TF-IDF", "Coseno",
                                        "ID y título BM25", "Score BM25"])

for id_q, (texto, _) in consultas.items():
    print(f"{id_q}: {texto}")
    display(tabla_comparacion(id_q))

Q1: love


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,1041 - Shakespeare's Sonnets,0.1275,1041 - Shakespeare's Sonnets,0.9582
1,2,16739 - The Greatest Thing In the World and,0.0720,29896 - Sex--The Unknown Quantity: The Spir,0.9579
2,3,67355 - Ioläus: An Anthology of Friendship,0.0656,16739 - The Greatest Thing In the World and,0.9577
3,4,18871 - A Channel Passage and Other Poems,0.0653,67355 - Ioläus: An Anthology of Friendship,0.9576
4,5,7550 - Quotes and Images From The Works of,0.0628,18871 - A Channel Passage and Other Poems,0.9570
5,6,29896 - Sex--The Unknown Quantity: The Spir,0.0530,42753 - The Expositor's Bible: The Epistles,0.9568
6,7,27851 - A Christmas Faggot,0.0512,33943 - Woman,0.9565
7,8,33943 - Woman,0.0498,"15878 - The Stepmother, A Drama in Five Act",0.9554
8,9,54058 - Love Poems and Others,0.0467,46272 - The Journal of Leo Tolstoi (First V,0.9545
9,10,71792 - Cobwebs to catch flies,0.0451,52243 - Daughters of Belgravia; vol. 1 of 3,0.9544


Q2: the election of the new president


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,65442 - Disunion and Restoration in Tenness,0.5250,27348 - The 2007 CIA World Factbook,6.7379
1,2,37737 - A History of the Republican Party,0.5248,37737 - A History of the Republican Party,6.7252
2,3,74395 - A short history of Belgium,0.5161,65442 - Disunion and Restoration in Tenness,6.6850
3,4,11021 - A Compilation of the Messages and P,0.5071,6812 - Abraham Lincoln: a History — Volume,6.6685
4,5,"17326 - History of Egypt, Chaldæa, Syria, B",0.5026,11021 - A Compilation of the Messages and P,6.6660
5,6,"49711 - The National Geographic Magazine, V",0.5007,17136 - Selected Official Documents of the,6.5982
6,7,35678 - France,0.4986,10291 - In the Fourth Year: Anticipations o,6.5534
7,8,"33127 - Encyclopaedia Britannica, 11th Edit",0.4986,53228 - Eclectic Magazine of Foreign Litera,6.5012
8,9,25842 - Royal Palaces and Parks of France,0.4982,20816 - Socialism as it is,6.4590
9,10,"64325 - Landscape in History, and Other Ess",0.4981,7382 - The Life of the Rt. Hon. Sir Charle,6.4122


Q3: ship captain storm sea


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,70148 - Time for survival,0.1135,21749 - Man on the Ocean: A Book about Boat,5.9910
1,2,72974 - Recollections of James Anthony Gard,0.0767,14172 - Willis the Pilot : A Sequel to the,5.9679
2,3,40580 - A general history of the pyrates,0.0641,14149 - The Pilots of Pomona: A Story of th,5.9532
3,4,50908 - A Relic of the Revolution,0.0467,25993 - With Cochrane the Dauntless,5.9336
4,5,4994 - Five Thousand Miles Underground; Or,0.0449,50470 - The Long Journey,5.9267
5,6,28437 - It's a Small Solar System,0.0449,24454 - Across the Spanish Main: A Tale of,5.9057
6,7,59309 - Forced Move,0.0422,69718 - Tall tales of Cape Cod,5.8735
7,8,35427 - The Raid of John Brown at Harper's,0.0402,"60506 - Powhatan; A Metrical Romance, in Se",5.8643
8,9,7509 - The Logbooks of the Lady Nelson,0.0389,72974 - Recollections of James Anthony Gard,5.8584
9,10,21749 - Man on the Ocean: A Book about Boat,0.0388,51441 - Third Reader: The Alexandra Readers,5.8521


... (6 outputs left out)

Q7: the history of the roman empire and its emperors in ancient times


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,35461 - A Short History of the World,0.4751,6839 - The Old Roman World : the Grandeur,14.0503
1,2,6839 - The Old Roman World : the Grandeur,0.4744,35461 - A Short History of the World,13.7289
2,3,"68185 - The passing of the great race; or,",0.4587,76054 - The government of the Ottoman Empir,13.5765
3,4,"42048 - Encyclopaedia Britannica, 11th Edit",0.4573,26639 - A Brief Commentary on the Apocalyps,13.3831
4,5,26639 - A Brief Commentary on the Apocalyps,0.4555,1972 - History of the Britons (Historia Br,13.0440
5,6,"33127 - Encyclopaedia Britannica, 11th Edit",0.4554,22257 - Bulgaria,13.0062
6,7,64029 - Life at the Zoo: Notes and Traditio,0.4524,"42048 - Encyclopaedia Britannica, 11th Edit",12.9823
7,8,76054 - The government of the Ottoman Empir,0.4515,"68185 - The passing of the great race; or,",12.9446
8,9,35678 - France,0.4496,74395 - A short history of Belgium,12.7532
9,10,74395 - A short history of Belgium,0.4478,"33127 - Encyclopaedia Britannica, 11th Edit",12.5580


Q8: whale harpoon


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,75370 - David goes to Greenland,0.0180,75370 - David goes to Greenland,13.8300
1,2,68177 - Among unknown Eskimo,0.0099,68177 - Among unknown Eskimo,13.6200
2,3,17254 - The Slant Book,0.0092,6862 - The Belted Seas,12.1041
3,4,69718 - Tall tales of Cape Cod,0.0076,14149 - The Pilots of Pomona: A Story of th,11.2744
4,5,6862 - The Belted Seas,0.0072,14172 - Willis the Pilot : A Sequel to the,10.8398
5,6,14149 - The Pilots of Pomona: A Story of th,0.0034,74431 - The Wampanoags in the seventeenth c,9.6043
6,7,34452 - The Iron Arrow Head or The Buckler,0.0033,34452 - The Iron Arrow Head or The Buckler,7.7220
7,8,"57069 - Obesity, or Excessive Corpulence: T",0.0033,19739 - Modern American Prose Selections,7.0839
8,9,4994 - Five Thousand Miles Underground; Or,0.0032,29765 - Webster's Unabridged Dictionary,5.9728
9,10,14172 - Willis the Pilot : A Sequel to the,0.0026,69718 - Tall tales of Cape Cod,5.1835


In [10]:
def resumen_comparacion():
    # Tamaños reales, |T_TFIDF ∩ T_BM25|, IDs en la misma posición y primer documento
    filas = []
    for id_q in consultas:
        ids_t = [d for d, _ in rankings[id_q]["tfidf"]]
        ids_m = [d for d, _ in rankings[id_q]["bm25"]]
        comunes = set(ids_t) & set(ids_m)
        misma_pos = sum(1 for a, b in zip(ids_t, ids_m) if a == b)
        filas.append((id_q, len(ids_t), len(ids_m), len(comunes), misma_pos,
                      ids_t[0] if ids_t else "-", ids_m[0] if ids_m else "-",
                      bool(ids_t and ids_m and ids_t[0] == ids_m[0])))
    return pd.DataFrame(filas, columns=["Consulta", "Tamaño TF-IDF", "Tamaño BM25", "|T ∩ B|",
                                        "Misma posición", "Primero TF-IDF", "Primero BM25", "¿Mismo primero?"])

display(resumen_comparacion())

,Consulta,Tamaño TF-IDF,Tamaño BM25,|T ∩ B|,Misma posición,Primero TF-IDF,Primero BM25,¿Mismo primero?
0,Q1,10,10,6,1,1041,1041,True
1,Q2,10,10,3,1,65442,27348,False
2,Q3,10,10,2,0,70148,21749,False
3,Q4,10,10,5,0,53639,59171,False
4,Q5,10,10,2,0,65794,23100,False
5,Q6,10,10,8,2,59428,59428,True
6,Q7,10,10,8,0,35461,6839,False
7,Q8,10,10,7,3,75370,75370,True


In [11]:
from itertools import combinations

idx_de = {d: i for i, d in enumerate(doc_ids)}

def posiciones(id_q):
    # Posición (1..10) de cada documento en cada top 10
    pt = {d: p for p, (d, _) in enumerate(rankings[id_q]["tfidf"], start=1)}
    pb = {d: p for p, (d, _) in enumerate(rankings[id_q]["bm25"], start=1)}
    return pt, pb

def buscar_candidatos(id_q, maximo=5):
    # Pares de documentos presentes en ambos top 10 cuyo orden relativo se invierte
    pt, pb = posiciones(id_q)
    candidatos = []
    for a, b in combinations(sorted(set(pt) & set(pb)), 2):
        dif_t, dif_b = pt[a] - pt[b], pb[a] - pb[b]
        if dif_t * dif_b < 0:
            candidatos.append((abs(dif_t - dif_b), a, b))
    candidatos.sort(reverse=True)
    filas = [(id_q, a, titulo_de[a][:25], pt[a], pb[a], int(longitudes[idx_de[a]]),
              b, titulo_de[b][:25], pt[b], pb[b], int(longitudes[idx_de[b]]))
             for _, a, b in candidatos[:maximo]]
    return pd.DataFrame(filas, columns=["Consulta", "Doc A", "Título A", "Pos TF-IDF A", "Pos BM25 A", "|d| A",
                                        "Doc B", "Título B", "Pos TF-IDF B", "Pos BM25 B", "|d| B"])

for id_q in consultas:
    display(buscar_candidatos(id_q, maximo=3))

,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q1,29896,Sex--The Unknown Quantity,6,2,58894,67355,Ioläus: An Anthology of F,3,4,40889
1,Q1,18871,A Channel Passage and Oth,4,5,24814,29896,Sex--The Unknown Quantity,6,2,58894
2,Q1,16739,The Greatest Thing In the,2,3,33168,29896,Sex--The Unknown Quantity,6,2,58894


,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q2,37737,A History of the Republic,2,2,83528,65442,Disunion and Restoration,1,3,22115


,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q3,21749,Man on the Ocean: A Book,10,1,53079,72974,Recollections of James An,2,9,88761


,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q4,16835,An Interpretation of Rudo,4,8,57090,29896,Sex--The Unknown Quantity,9,3,58894
1,Q4,29896,Sex--The Unknown Quantity,9,3,58894,77075,Man's supreme inheritance,5,6,82437
2,Q4,16835,An Interpretation of Rudo,4,8,57090,77075,Man's supreme inheritance,5,6,82437


,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q5,1041,Shakespeare's Sonnets,6,8,17677,71332,History in English words,9,3,59815


,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q6,37149,"Fritz to the Front, or, t",3,9,27068,72855,The unpleasantness at the,8,4,72188
1,Q6,37149,"Fritz to the Front, or, t",3,9,27068,52498,No. 13 Toroni,7,6,59955
2,Q6,26176,The Secret House,4,3,59082,37149,"Fritz to the Front, or, t",3,9,27068


,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q7,68185,The passing of the great,3,8,120450,76054,The government of the Ott,8,3,126237
1,Q7,33127,"Encyclopaedia Britannica,",6,10,166284,76054,The government of the Ott,8,3,126237
2,Q7,42048,"Encyclopaedia Britannica,",4,7,182288,76054,The government of the Ott,8,3,126237


,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q8,14172,Willis the Pilot : A Sequ,10,5,112349,69718,Tall tales of Cape Cod,4,10,20989
1,Q8,6862,The Belted Seas,5,3,50457,69718,Tall tales of Cape Cod,4,10,20989
2,Q8,14149,The Pilots of Pomona: A S,6,4,86062,69718,Tall tales of Cape Cod,4,10,20989


In [12]:
CONSULTA_ELEGIDA = "Q3"
candidatos = buscar_candidatos(CONSULTA_ELEGIDA, maximo=5)
display(candidatos)

FILA = 0
DOC_A, DOC_B = candidatos.loc[FILA, "Doc A"], candidatos.loc[FILA, "Doc B"]

def desglose_pareja(id_q, doc_a, doc_b):
    # Por término: df, f(t,d), |d|, aporte BM25 y aporte al coseno, para dos documentos
    tokens = terminos_en_vocabulario(preprocess(consultas[id_q][0]))
    conteos = Counter(tokens)
    terminos = list(dict.fromkeys(tokens))
    q_w = {t: conteos[t] * idf_tfidf_vec[indice_vocab[t]] for t in terminos}
    norma_q = math.sqrt(sum(v * v for v in q_w.values()))
    pt, pb = posiciones(id_q)
    score_bm25 = dict(rankings[id_q]["bm25"])
    score_cos = dict(rankings[id_q]["tfidf"])

    filas, resumen = [], []
    for doc in (doc_a, doc_b):
        i = idx_de[doc]
        total_bm25 = total_cos = 0.0
        for t in terminos:
            j = indice_vocab[t]
            f = TF[i, j]
            aporte = (idf_bm25_vec[j] * f * (K1 + 1) /
                      (f + K1 * (1 - B + B * longitudes[i] / avgdl))) if f > 0 else 0.0
            aporte_cos = q_w[t] * X_tfidf[i, j] / (norma_q * normas_docs[i])
            total_bm25 += aporte
            total_cos += aporte_cos
            filas.append((doc, t, df[t], round(idf_bm25_vec[j], 3), int(f),
                          int(longitudes[i]), round(aporte, 4), round(aporte_cos, 4)))
        resumen.append((doc, titulo_de[doc][:30], int(longitudes[i]), round(longitudes[i] / avgdl, 2),
                        pt.get(doc), round(total_cos, 4), pb.get(doc), round(total_bm25, 4)))
        if doc in score_bm25:
            assert abs(total_bm25 - score_bm25[doc]) < 1e-9
        if doc in score_cos:
            assert abs(total_cos - score_cos[doc]) < 1e-9

    print(f"{id_q}: {consultas[id_q][0]} | avgdl = {avgdl:,.1f}")
    display(pd.DataFrame(filas, columns=["Doc", "Término", "df(t)", "idf BM25", "f(t,d)", "|d|",
                                         "Aporte BM25", "Aporte coseno"]))
    display(pd.DataFrame(resumen, columns=["Doc", "Título", "|d|", "|d|/avgdl", "Pos TF-IDF",
                                           "Coseno", "Pos BM25", "Score BM25"]))

desglose_pareja(CONSULTA_ELEGIDA, DOC_A, DOC_B)

,Consulta,Doc A,Título A,Pos TF-IDF A,Pos BM25 A,|d| A,Doc B,Título B,Pos TF-IDF B,Pos BM25 B,|d| B
0,Q3,21749,Man on the Ocean: A Book,10,1,53079,72974,Recollections of James An,2,9,88761


Q3: ship captain storm sea | avgdl = 63,308.1


,Doc,Término,df(t),idf BM25,"f(t,d)",|d|,Aporte BM25,Aporte coseno
0,21749,ship,457,0.772,176,53079,1.6881,0.0218
1,21749,captain,433,0.826,44,53079,1.7743,0.0058
2,21749,storm,479,0.725,26,53079,1.5327,0.0031
3,21749,sea,626,0.458,98,53079,0.9959,0.0082
4,72974,ship,457,0.772,222,88761,1.6864,0.0192
5,72974,captain,433,0.826,581,88761,1.8119,0.0533
6,72974,storm,479,0.725,10,88761,1.3795,0.0008
7,72974,sea,626,0.458,59,88761,0.9807,0.0034


,Doc,Título,|d|,|d|/avgdl,Pos TF-IDF,Coseno,Pos BM25,Score BM25
0,21749,Man on the Ocean: A Book about,53079,0.84,10,0.0388,1,5.9910
1,72974,Recollections of James Anthony,88761,1.40,2,0.0767,9,5.8584


### **Explicación.** 
Elegí los documentos 21749 (*Man on the Ocean*, 0.84 veces la longitud media) y 72974 (*Recollections of James Anthony Gardner*, 1.40 veces la media). TF-IDF pone a 72974 en el puesto 2 (coseno 0.0767) y a 21749 en el 10 (0.0388). BM25 los invierte: 21749 queda primero (5.9910) y 72974 noveno (5.8584).

En TF-IDF todo se explica por `captain`: aparece 581 veces en 72974 y solo 44 en 21749, y como el peso crece de forma lineal con la frecuencia, su aporte al coseno es 0.0533 frente a 0.0058. Esa diferencia por sí sola es mayor que la diferencia total entre los dos documentos.

BM25 no cae en eso porque satura la frecuencia: con 13 veces más apariciones, `captain` aporta apenas 1.8119 frente a 1.7743. Lo que decide es qué documento cubre mejor los cuatro términos. 21749 menciona `storm` 26 veces (aporte 1.5327) y 72974 solo 10 (1.3795), una diferencia de 0.15 que, sumada a `sea` y `ship`, compensa lo que pierde en `captain` y da la ventaja total de 0.1326. A esto se suma la longitud: 72974 es más largo que la media, y su denominador penaliza cada término, mientras que 21749 es más corto y casi no se penaliza.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [13]:
def perfil_consulta(id_q, k=10):
    # Rango de longitudes relativas y de frecuencia total de términos en el top 10 BM25 base
    tokens = preprocess(consultas[id_q][0])
    cols = [indice_vocab[t] for t in dict.fromkeys(terminos_en_vocabulario(tokens))]
    top = ordenar_top(scores_bm25(tokens), k)
    if not top:
        return (id_q, 0, None, None, None, None)
    filas_idx = [idx_de[d] for d, _ in top]
    ratios = longitudes[filas_idx] / avgdl
    f_tot = np.asarray(TF[filas_idx][:, cols].sum(axis=1)).ravel()
    return (id_q, len(top), round(ratios.min(), 2), round(ratios.max(), 2),
            int(f_tot.min()), int(f_tot.max()))

display(pd.DataFrame([perfil_consulta(q) for q in consultas],
                     columns=["Consulta", "Docs en top 10", "|d|/avgdl mín", "|d|/avgdl máx",
                              "f total mín", "f total máx"]))

,Consulta,Docs en top 10,|d|/avgdl mín,|d|/avgdl máx,f total mín,f total máx
0,Q1,10,0.28,1.53,95,318
1,Q2,10,0.29,16.12,3293,69106
2,Q3,10,0.33,2.05,89,872
3,Q4,10,0.21,1.90,33,183
4,Q5,10,0.28,3.44,5,64
5,Q6,10,0.43,1.25,19,110
6,Q7,10,0.21,3.15,2742,43281
7,Q8,10,0.33,73.20,2,181


In [14]:
CONSULTA_PARAMS = "Q3"

CONFIGS = {
    "Base": (1.2, 0.75),
    "b=0": (1.2, 0.0),       # sin ajuste por longitud
    "k1=2": (2.0, 0.75),     # mayor efecto de las repeticiones
}

tokens_p = preprocess(consultas[CONSULTA_PARAMS][0])
terminos_p = list(dict.fromkeys(terminos_en_vocabulario(tokens_p)))
cols_p = [indice_vocab[t] for t in terminos_p]

# Ranking completo
pos_conf, score_conf = {}, {}
for nombre, (k1, b) in CONFIGS.items():
    lista = ordenar_top(scores_bm25(tokens_p, k1, b), k=N)
    pos_conf[nombre] = {d: p for p, (d, _) in enumerate(lista, start=1)}
    score_conf[nombre] = dict(lista)

# Documentos a seguir, los que aparecen en el top 10 de alguna configuración
seguir = set()
for nombre in CONFIGS:
    seguir |= {d for d, p in pos_conf[nombre].items() if p <= 10}
seguir = sorted(seguir, key=lambda d: pos_conf["Base"].get(d, N + 1))

filas = []
for d in seguir:
    i = idx_de[d]
    p_base, p_b0, p_k2 = (pos_conf[n].get(d) for n in CONFIGS)
    filas.append((d, titulo_de[d][:30], int(longitudes[i]), round(longitudes[i] / avgdl, 2),
                  int(TF[i, cols_p].sum()), p_base, p_b0, p_k2,
                  p_b0 - p_base if p_b0 and p_base else None,
                  p_k2 - p_base if p_k2 and p_base else None))

tabla_pos = pd.DataFrame(filas, columns=["Doc", "Título", "|d|", "|d|/avgdl", "f total",
                                         "Pos Base", "Pos b=0", "Pos k1=2", "Δ b=0", "Δ k1=2"])
print(f"{CONSULTA_PARAMS}: {consultas[CONSULTA_PARAMS][0]}  (Δ negativo = el documento sube respecto a Base)")
display(tabla_pos)

# Scores de los mismos documentos en las tres configuraciones
display(pd.DataFrame([(d, *(round(score_conf[n].get(d, 0.0), 4) for n in CONFIGS)) for d in seguir],
                     columns=["Doc"] + [f"Score {n}" for n in CONFIGS]))

Q3: ship captain storm sea  (Δ negativo = el documento sube respecto a Base)


,Doc,Título,|d|,|d|/avgdl,f total,Pos Base,Pos b=0,Pos k1=2,Δ b=0,Δ k1=2
0,21749,Man on the Ocean: A Book about,53079,0.84,344,1,6,1,5,0
1,14172,Willis the Pilot : A Sequel to,112349,1.77,373,2,2,2,0,0
2,14149,The Pilots of Pomona: A Story,86062,1.36,418,3,5,3,2,0
3,25993,With Cochrane the Dauntless,129914,2.05,583,4,3,4,-1,0
4,50470,The Long Journey,33813,0.53,139,5,22,5,17,0
5,24454,Across the Spanish Main: A Tal,93967,1.48,369,6,8,6,2,0
6,69718,Tall tales of Cape Cod,20989,0.33,96,7,39,7,32,0
7,60506,"Powhatan; A Metrical Romance,",32776,0.52,89,8,31,9,23,1
8,72974,Recollections of James Anthony,88761,1.40,872,9,14,8,5,-1
9,51441,Third Reader: The Alexandra Re,76175,1.20,164,10,15,12,5,2


,Doc,Score Base,Score b=0,Score k1=2
0,21749,5.9910,5.9743,8.0608
1,14172,5.9679,6.0215,8.0095
2,14149,5.9532,5.9866,7.9813
3,25993,5.9336,6.0114,7.9414
4,50470,5.9267,5.8317,7.9219
5,24454,5.9057,5.9592,7.8811
6,69718,5.8735,5.6578,7.8104
7,60506,5.8643,5.7320,7.7855
8,72974,5.8584,5.9127,7.7957
9,51441,5.8521,5.8856,7.7586


In [15]:
def aportes_documento(doc):
    # Aporte BM25 de cada término de la consulta al documento, en las tres configuraciones
    i = idx_de[doc]
    filas, totales = [], {n: 0.0 for n in CONFIGS}
    for t in terminos_p:
        j = indice_vocab[t]
        f = TF[i, j]
        fila = [t, df[t], int(f)]
        for nombre, (k1, b) in CONFIGS.items():
            ap = (idf_bm25_vec[j] * f * (k1 + 1) /
                  (f + k1 * (1 - b + b * longitudes[i] / avgdl))) if f > 0 else 0.0
            totales[nombre] += ap
            fila.append(round(ap, 4))
        filas.append(fila)
    for nombre in CONFIGS:                                     # control: coincide con el score del ranking
        assert abs(totales[nombre] - score_conf[nombre].get(doc, 0.0)) < 1e-9
    tabla = pd.DataFrame(filas, columns=["Término", "df(t)", "f(t,d)"] + [f"Aporte {n}" for n in CONFIGS])
    tabla.loc[len(tabla)] = ["TOTAL", "", ""] + [round(totales[n], 4) for n in CONFIGS]
    return tabla

DOCS_EXPLICAR = None

if DOCS_EXPLICAR is None:
    DOCS_EXPLICAR = []
    for col in ["Δ b=0", "Δ k1=2"]:
        d = tabla_pos.loc[tabla_pos[col].abs().fillna(0).idxmax(), "Doc"]
        if d not in DOCS_EXPLICAR:
            DOCS_EXPLICAR.append(d)
    movimiento = tabla_pos[["Δ b=0", "Δ k1=2"]].abs().fillna(0).sum(axis=1)
    estable = tabla_pos.loc[movimiento.idxmin(), "Doc"]
    if estable not in DOCS_EXPLICAR:
        DOCS_EXPLICAR.append(estable)

for d in DOCS_EXPLICAR:
    i = idx_de[d]
    print(f"Doc {d}: {titulo_de[d]} | |d| = {int(longitudes[i]):,} ({longitudes[i] / avgdl:.2f} x avgdl)")
    display(aportes_documento(d))

Doc 29765: Webster's Unabridged Dictionary | |d| = 4,634,329 (73.20 x avgdl)


,Término,df(t),"f(t,d)",Aporte Base,Aporte b=0,Aporte k1=2
0,ship,457,1435,1.6234,1.6968,2.1505
1,captain,433,106,1.1185,1.7965,1.2141
2,storm,479,197,1.1938,1.5853,1.3942
3,sea,626,1761,0.9702,1.0059,1.2918
4,TOTAL,,,4.9058,6.0845,6.0505


Doc 10: The King James Version of the Bible | |d| = 790,393 (12.48 x avgdl)


,Término,df(t),"f(t,d)",Aporte Base,Aporte b=0,Aporte k1=2
0,ship,457,71,1.4609,1.6700,1.8223
1,captain,433,139,1.6776,1.8012,2.1764
2,storm,479,14,0.8744,1.4690,0.9164
3,sea,626,400,0.9784,1.0036,1.3097
4,TOTAL,,,4.9912,5.9439,6.2248


Doc 14172: Willis the Pilot : A Sequel to the Swiss Family Robinson | |d| = 112,349 (1.77 x avgdl)


,Término,df(t),"f(t,d)",Aporte Base,Aporte b=0,Aporte k1=2
0,ship,457,92,1.6639,1.6764,2.2388
1,captain,433,121,1.7887,1.7989,2.4143
2,storm,479,41,1.5244,1.5496,2.0192
3,sea,626,119,0.9908,0.9966,1.3371
4,TOTAL,,,5.9679,6.0215,8.0095


### **Explicación.**
`b = 0` (sin ajuste por longitud).* Cambia mucho el ranking, y en la dirección esperada: favorece a los documentos largos y perjudica a los cortos. El diccionario de Webster (73.2 veces la longitud media) sube del puesto 113 al 1 y su score pasa de 4.9058 a 6.0845; por ejemplo, el aporte de `captain` (106 apariciones) sube de 1.1185 a 1.7965, casi el máximo posible. La Biblia (12.5 veces la media) sube del 102 al 10, con `storm` pasando de 0.8744 a 1.4690. A cambio, los documentos cortos caen: 50470 (0.53 veces la media) baja del 5 al 22, 69718 (0.33) del 7 al 39 y 21749 (0.84) del 1 al 6. Con `b = 0` un documento largo ya no paga por serlo y alcanza fácilmente el tope en los cuatro términos.

`k1 = 2` (más efecto de las repeticiones).* Sube el tope de cada aporte, de idf × 2.2 a idf × 3, y la saturación llega más tarde. Los scores aumentan en casi todos los documentos (en el top, de unos 5.9 a unos 8.0), pero el orden casi no cambia (máximo 4 puestos). Los que ganan son los que repiten mucho los términos: *A general history of the pyrates* (1133 apariciones en total) pasa del 13 al 10, y la Biblia del 102 al 98.

*Un documento que no se mueve.* 14172 (*Willis the Pilot*) está en el puesto 2 en las tres configuraciones, aunque su score cambia de 5.9679 a 6.0215 con `b = 0` y a 8.0095 con `k1 = 2`. Tiene los cuatro términos con frecuencias altas (entre 41 y 121 apariciones), así que ya estaba cerca del tope y `b` casi no lo afecta (el aporte de `ship` va de 1.6639 a 1.6764). Con `k1 = 2` sube unos 2 puntos, pero sus vecinos suben lo mismo (21749 pasa de 5.9910 a 8.0608 y 14149 de 5.9532 a 7.9813), así que las distancias entre ellos se mantienen y el orden no cambia.

En resumen, `b` controla cuánto pesa la longitud del documento y es el que más reordena el ranking. `k1` controla cuánto vale repetir un término, y sobre todo cambia la escala de los scores.